In [13]:
import requests
from bs4 import BeautifulSoup
import fitz  
import re
import io
import pandas as pd
from datetime import datetime
from urllib.parse import urljoin

In [18]:
class DataCentreIntelligence:
    def __init__(self):
        self.results = []
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/121.0.0.0'
        }
        self.base_url = "https://www.slough.gov.uk"

    def clean_url(self, url, parent_url):
        """Heals broken council links and handles relative paths."""
        # Fix the specific Slough CMS typo (missing slash)
        if "slough.gov.ukuser_uploads" in url:
            url = url.replace("slough.gov.ukuser_uploads", "slough.gov.uk/user_uploads")
        
        # Handle relative paths (e.g., user_uploads/file.pdf)
        if not url.startswith('http'):
            # If it's a Citizen Space link, join with the parent; otherwise use base Slough URL
            if "citizenspace.com" in parent_url:
                return urljoin(parent_url, url)
            return urljoin(self.base_url, url)
        return url

    def extract_metrics(self, url, doc_name):
        """Scans PDF text for MW, SQM, and Substation markers."""
        metrics = {"MW": "N/A", "SQM": "N/A", "Substation": "No"}
        try:
            response = requests.get(url, headers=self.headers, timeout=15)
            response.raise_for_status()
            
            with fitz.open(stream=io.BytesIO(response.content), filetype="pdf") as doc:
                text = "".join([page.get_text() for page in doc])
                
                # Regex patterns for Data Centre specific KPIs
                mw_match = re.search(r'(\d+(?:\.\d+)?)\s*(?:MW|MVA|megawatt)', text, re.IGNORECASE)
                sqm_match = re.search(r'(\d{1,3}(?:,\d{3})*(?:\.\d+)?)\s*(?:sqm|sq m|square metre)', text, re.IGNORECASE)
                sub_match = re.search(r'(substation|132kv|grid supply point|National Grid)', text, re.IGNORECASE)
                
                if mw_match: metrics["MW"] = mw_match.group(0)
                if sqm_match: metrics["SQM"] = sqm_match.group(0)
                if sub_match: metrics["Substation"] = "Yes"
        except Exception as e:
            # Silently log errors for the dashboard
            pass 
        return metrics

    def survey_site(self, url):
        print(f"Checking Source: {url}")
        
        # Case 1: Direct PDF Link
        if url.lower().endswith('.pdf') or '/file/' in url:
            title = url.split('/')[-1]
            metrics = self.extract_metrics(url, title)
            self.results.append({**{"Source": "Direct PDF", "Title": title}, **metrics, "URL": url})
            
        # Case 2: Hub Page (Crawler)
        else:
            try:
                response = requests.get(url, headers=self.headers, timeout=15)
                soup = BeautifulSoup(response.text, 'html.parser')
                # Find all links that look like downloads or PDFs
                links = [a for a in soup.find_all('a', href=True) if '.pdf' in a['href'].lower() or 'download' in a.text.lower()]
                
                for link in links[:10]: # Scan first 10 documents
                    file_url = self.clean_url(link['href'], url)
                    title = link.text.strip() or "Linked Document"
                    metrics = self.extract_metrics(file_url, title)
                    self.results.append({**{"Source": "Hub Page", "Title": title}, **metrics, "URL": file_url})
            except Exception as e:
                print(f"  [!] Web Error: {e}")

    def generate_dashboard(self):
        if not self.results:
            print("No data captured.")
            return

        df = pd.DataFrame(self.results)
        
        # Priority Scoring Logic (High MW/SQM = High Priority)
        def score_row(row):
            score = 0
            if row['MW'] != 'N/A': score += 3
            if row['SQM'] != 'N/A': score += 2
            if row['Substation'] == 'Yes': score += 1
            return score

        df['Score'] = df.apply(score_row, axis=1)
        df['Status'] = df['Score'].apply(lambda s: "CRITICAL" if s >= 5 else ("HIGH" if s >= 3 else ("MEDIUM" if s >= 1 else "MONITOR")))
        
        # Sort by most important first
        df = df.sort_values(by='Score', ascending=False)

        print(f"\nEXECUTIVE DASHBOARD: SLOUGH DATA CENTRE OPPORTUNITIES\n")
        print(df[['Status', 'Title', 'MW', 'SQM', 'Substation']].to_string(index=False))
        
        df.to_csv("slough_opportunity_report.csv", index=False)
        print(f"\nFull dataset saved to 'slough_opportunity_report.csv' at {datetime.now().strftime('%H:%M')}")


In [19]:
if __name__ == "__main__":
    # The 'Target Set' for the Slough Cluster
    targets = [
        "https://slough.citizenspace.com/planning-policy/spz/", 
        "https://www.slough.gov.uk/downloads/file/4813/ldf-annual-monitoring-report-2024-25", 
        "https://slough.moderngov.co.uk/documents/s74693/Spatial%20Strategy%20Appendix.pdf"
    ]
    
    engine = DataCentreIntelligence()
    for t in targets:
        engine.survey_site(t)
    engine.generate_dashboard()

Checking Source: https://slough.citizenspace.com/planning-policy/spz/
Checking Source: https://www.slough.gov.uk/downloads/file/4813/ldf-annual-monitoring-report-2024-25
Checking Source: https://slough.moderngov.co.uk/documents/s74693/Spatial%20Strategy%20Appendix.pdf

EXECUTIVE DASHBOARD: SLOUGH DATA CENTRE OPPORTUNITIES

  Status                                                                  Title          MW                  SQM Substation
CRITICAL                           SPZ (Amended) EIA Screening Report May24.pdf 50 megawatt 698,830 square metre        Yes
    HIGH                                            SPZ  Deposit Written Scheme         N/A          600,000 sqm        Yes
    HIGH                     SPZ-Deposit Consultation - Appendices (merged).pdf         N/A            1,000 sqm        Yes
  MEDIUM                                                       download the PDF         N/A          55,000 sq m         No
  MEDIUM                                               